# OpenAI API with Python

This notebook shows a minimal way to call the OpenAI API from Python using the current `openai` SDK.


In [8]:
%pip install -q openai python-dotenv


Note: you may need to restart the kernel to use updated packages.


## API key setup

Preferred approach: keep `OPENAI_API_KEY` in a repo-local `.env` file or set it in your environment before starting Jupyter.

This notebook will try to load `.env` automatically.

PowerShell for the current terminal session:

```powershell
$env:OPENAI_API_KEY = "your_api_key_here"
```

If the key is still missing after loading `.env`, the next cell will prompt for it securely.


In [15]:
import os
from pathlib import Path
from getpass import getpass

from dotenv import load_dotenv

# Load environment variables from the repo root if a .env file exists there.
repo_root = Path.cwd()
load_dotenv(repo_root / ".env", override=False)

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")

MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
MODEL = 'gpt-5-nano'
print(f"Using model: {MODEL}")


Using model: gpt-5-nano


In [16]:
from openai import OpenAI

client = OpenAI()


## Minimal API call

The `Responses API` is the primary API in the current Python SDK.


In [17]:
response = client.responses.create(
    model=MODEL,
    input="Who is Tom Cruise's mother?",
)

print(response.output_text)


Mary Lee Pfeiffer. She was Tom Cruise's mother (she later also went by Mary Lee Mapother after marriage) and was a special education teacher.


In [ ]:
response = client.responses.create(
    model=MODEL,
    input="Who is Mary Lee Pfeiffer's son?",
)

print(response.output_text)

Tom Cruise (Thomas Cruise Mapother IV).


## Reusable helper

This wraps the API call with basic error handling so you can reuse it for different prompts.


In [5]:
from openai import APIConnectionError, APIStatusError, RateLimitError

def ask_openai(prompt: str, model: str = MODEL, instructions: str | None = None) -> str:
    try:
        response = client.responses.create(
            model=model,
            instructions=instructions,
            input=prompt,
        )
        return response.output_text
    except RateLimitError as exc:
        return f"Rate limit error: {exc}"
    except APIConnectionError as exc:
        return f"Connection error: {exc}"
    except APIStatusError as exc:
        return f"API status error {exc.status_code}, request id={exc.request_id}: {exc}"


In [6]:
reply = ask_openai(
    prompt="Explain gradient descent in 3 bullet points.",
    instructions="You are a concise machine learning tutor.",
)

print(reply)


- **Objective**: Gradient descent is an optimization algorithm used to minimize the loss function in machine learning models by iteratively adjusting parameters.
  
- **Direction of Update**: It calculates the gradient (or slope) of the loss function concerning the model parameters, determining the steepest descent direction to reduce the loss.

- **Learning Rate**: The size of the steps taken towards the minimum is controlled by a hyperparameter called the learning rate, which can affect convergence speed and stability.


## Optional: Chat Completions example

The older Chat Completions API is still supported by the SDK.


In [7]:
completion = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "developer", "content": "You are a helpful ML tutor."},
        {"role": "user", "content": "What is overfitting?"},
    ],
)

print(completion.choices[0].message.content)


Overfitting is a common problem in machine learning and statistical modeling, where a model learns not only the underlying patterns in the training data but also the noise and random fluctuations present in that data. This often results in a model that performs very well on the training dataset but poorly on unseen or validation data.

Here are some key points to understand about overfitting:

1. **Complexity**: Overfitting typically occurs when a model is too complex relative to the amount of training data available. Complex models, such as deep neural networks with many layers, can capture intricate patterns in the training data, but they can also capture irrelevant noise.

2. **Training vs. Testing Performance**: An overfitted model will usually have high training accuracy but significantly lower accuracy on validation/testing data. This discrepancy indicates that the model has learned specific details of the training dataset that do not generalize to new, unseen data.

3. **Symptom